# 03 — Evaluation contract

This notebook freezes the ruler before comparing models. It physically
separates development and holdout truth and runs nine small adversarial tests.

It does not create features, train models or search thresholds.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT")
    or os.getenv("ANOMALY_DRIVE_ROOT")
    or (
        "/content/drive/MyDrive/anomaly_detection"
        if IN_COLAB else Path.home() / "anomaly_detection_data"
    )
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB
    else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
    else Path.cwd() / "notebooks" / "drive_research",
)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

SECTOR = os.getenv("ANOMALY_SECTOR", "telecom")
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_10_1_run1",
    "petrobras_3w": "petrobras_3w_core_v0_10_1_run1",
}
if SECTOR not in CANONICAL_RUN_IDS:
    raise ValueError(f"Choose one of {list(CANONICAL_RUN_IDS)}")

import tempfile

from milestone1_core import CORE_VERSION, new_output_directory, read_json, write_json
from evaluation_core import (
    ALERT_COLUMNS, EVALUATION_CORE_VERSION, SCORE_COLUMNS,
    evaluate_alerts, partition_truth, scores_to_alerts,
)

EDA_VERSION = "1.3.0"
EVALUATION_VERSION = "1.3.0"
CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
EDA_RUN_ID = os.getenv("EDA_RUN_ID", f"{SECTOR}_eda_v1_3_run1")
EVALUATION_RUN_ID = os.getenv("EVALUATION_RUN_ID", f"{SECTOR}_evaluation_v1_3_run1")
RUN_ROOT = DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / CANONICAL_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
EVAL_ROOT = RUN_ROOT / "SPEC-EVAL"
SPLIT_ROOT = RUN_ROOT / "SPLITS"
EDA_ROOT = DATA_ROOT / "outputs" / "eda" / f"v{EDA_VERSION}" / SECTOR / EDA_RUN_ID
OUTPUT_ROOT = DATA_ROOT / "outputs" / "evaluation" / f"v{EVALUATION_VERSION}" / SECTOR / EVALUATION_RUN_ID

POLICY = {
    "telecom": {
        "decision_horizon_seconds": 48 * 60 * 60,
        "primary_exposure_unit": "entity_day",
        "false_alert_budget": 0.01,
        "primary_selection_metric": "preimpact_event_recall",
        "grouping_window_seconds": 60 * 60,
    },
    "petrobras_3w": {
        "decision_horizon_seconds": 6 * 60 * 60,
        "primary_exposure_unit": "episode",
        "false_alert_budget": 0.10,
        "primary_selection_metric": "event_recall",
        "grouping_window_seconds": 0,
    },
}
display(pd.Series({
    "sector": SECTOR, "canonical_truth": str(EVAL_ROOT),
    "evaluation_output": str(OUTPUT_ROOT),
}, name="value").to_frame())

## 2. Partition truth without changing it

In [ ]:
if not EVAL_ROOT.is_dir():
    raise FileNotFoundError("This development source has no SPEC-EVAL")

manifest = read_json(CORE_ROOT / "manifest.json")
decisions = read_json(EDA_ROOT / "eda_decisions.json")
if manifest["sector"] != SECTOR:
    raise ValueError("Canonical sector does not match the requested sector")
if decisions["eda_version"] != EDA_VERSION:
    raise ValueError("EDA decisions are not from the declared EDA version")
canonical_build_ts = pd.Timestamp(
    (CORE_ROOT / "manifest.json").stat().st_mtime, unit="s", tz="UTC"
)
display(pd.Series({
    "canonical_run_id": CANONICAL_RUN_ID,
    "canonical_build_ts": canonical_build_ts,
}, name="value").to_frame())
registry = pd.read_parquet(CORE_ROOT / "entity_registry.parquet")
events = pd.read_parquet(EVAL_ROOT / "fault_events.parquet")
intervals = pd.read_parquet(EVAL_ROOT / "fault_entity_intervals.parquet")
condition_path = EVAL_ROOT / "condition_states.parquet"
conditions = pd.read_parquet(condition_path) if condition_path.is_file() else pd.DataFrame(
    columns=["entity_id", "start_ts", "end_ts", "condition_code", "label_source", "source_instance_id"]
)

for frame in (registry, events, intervals, conditions):
    for column in frame.columns:
        if column.endswith("_ts") or column in {"observed_from", "observed_to"}:
            frame[column] = pd.to_datetime(frame[column], utc=True, errors="coerce")

time_path = SPLIT_ROOT / "time_partitions.parquet"
entity_path = SPLIT_ROOT / "entity_partitions.parquet"
time_partitions = pd.read_parquet(time_path) if time_path.is_file() else pd.DataFrame()
entity_partitions = pd.read_parquet(entity_path) if entity_path.is_file() else pd.DataFrame()
for frame in (time_partitions,):
    for column in ("start_ts", "end_ts"):
        if column in frame:
            frame[column] = pd.to_datetime(frame[column], utc=True)
primary_split = "time" if not time_partitions.empty else "entity"

truth, audit, truth_summary = partition_truth(
    events, intervals, conditions, registry,
    primary_split=primary_split,
    time_partitions=time_partitions,
    entity_partitions=entity_partitions,
)
display(truth_summary)
print("Scoreable development faults:", len(truth["development"]["fault_events"]))
print("Sealed holdout faults:", len(truth["holdout"]["fault_events"]))

## 3. Nine contract tests

In [ ]:
BASE = pd.Timestamp("2025-01-01", tz="UTC")

test_events = pd.DataFrame([
    ("F1", "fault_a", "asset-1", BASE, BASE, BASE + pd.Timedelta(minutes=30), BASE + pd.Timedelta(hours=2), pd.NA, "test", "one"),
    ("F2", "fault_b", "asset-2", BASE + pd.Timedelta(hours=3), BASE + pd.Timedelta(hours=3), pd.NaT, BASE + pd.Timedelta(hours=5), pd.NA, "test", "two"),
], columns=["fault_id", "fault_type", "domain_id", "onset_ts", "observable_ts", "impact_ts", "end_ts", "group_id", "label_source", "source_instance_id"])
test_intervals = pd.DataFrame([
    ("F1", "asset-1", BASE, BASE + pd.Timedelta(hours=2), "test", "one"),
    ("F2", "asset-2", BASE + pd.Timedelta(hours=3), BASE + pd.Timedelta(hours=5), "test", "two"),
], columns=["fault_id", "entity_id", "start_ts", "end_ts", "label_source", "source_instance_id"])

def alert(alert_id, entity_id, when):
    return (alert_id, "test_model", entity_id, f"{entity_id}::episode", when,
            when + pd.Timedelta(minutes=1), when, 10.0, 2, "metric__change")

def metric(result, name):
    return result["metrics"].set_index("metric").loc[name, "value"]

perfect = pd.DataFrame([
    alert("A1", "asset-1", BASE + pd.Timedelta(minutes=10)),
    alert("A2", "asset-2", BASE + pd.Timedelta(hours=3, minutes=10)),
], columns=ALERT_COLUMNS)
perfect_result = evaluate_alerts(
    perfect, test_events, test_intervals,
    exposure_value=10, exposure_unit="entity_day",
    decision_horizon_seconds=3600,
)
assert metric(perfect_result, "event_recall") == 1
assert metric(perfect_result, "alert_precision") == 1

def score_frame(values):
    return pd.DataFrame({
        "event_ts": pd.date_range(BASE, periods=len(values), freq="1min"),
        "entity_id": "asset-1",
        "episode_id": "asset-1::episode",
        "anomaly_score": values, "model_id": "test_model",
    })[SCORE_COLUMNS]

empty = scores_to_alerts(
    score_frame([0, 0, 0, 0, 0]), threshold=1,
    min_consecutive=2, recovery_consecutive=2,
)
empty_result = evaluate_alerts(
    empty, test_events, test_intervals,
    exposure_value=10, exposure_unit="entity_day",
    decision_horizon_seconds=3600,
)
assert metric(empty_result, "event_recall") == 0

sustained = scores_to_alerts(
    score_frame([2, 2, 2, 2, 2]), threshold=1,
    min_consecutive=2, recovery_consecutive=2,
)
assert len(sustained) == 1

recovered = scores_to_alerts(
    score_frame([2, 2, 0, 2, 2, 0, 0, 2, 2]), threshold=1,
    min_consecutive=2, recovery_consecutive=2,
)
assert len(recovered) == 2
assert recovered["n_scores"].tolist() == [5, 2]

late = pd.DataFrame([
    alert("A1", "asset-1", BASE + pd.Timedelta(minutes=90)),
], columns=ALERT_COLUMNS)
late_result = evaluate_alerts(
    late, test_events.iloc[[0]], test_intervals.iloc[[0]],
    exposure_value=10, exposure_unit="entity_day",
    decision_horizon_seconds=3600,
)
assert metric(late_result, "event_recall") == 0

duplicate = pd.DataFrame([
    alert("A1", "asset-1", BASE + pd.Timedelta(minutes=10)),
    alert("A2", "asset-1", BASE + pd.Timedelta(minutes=20)),
], columns=ALERT_COLUMNS)
duplicate_result = evaluate_alerts(
    duplicate, test_events.iloc[[0]], test_intervals.iloc[[0]],
    exposure_value=10, exposure_unit="entity_day",
    decision_horizon_seconds=3600,
)
assert metric(duplicate_result, "event_recall") == 1
assert metric(duplicate_result, "duplicate_alerts") == 1

invalid = perfect.iloc[[0]].copy()
invalid["peak_ts"] = invalid["alert_end"] + pd.Timedelta(minutes=1)
try:
    evaluate_alerts(
        invalid, test_events.iloc[[0]], test_intervals.iloc[[0]],
        exposure_value=10, exposure_unit="entity_day",
        decision_horizon_seconds=3600,
    )
except ValueError:
    invalid_alert_test = "pass"
else:
    raise AssertionError("Invalid alert was accepted")

with tempfile.TemporaryDirectory() as temporary:
    mounted = Path(temporary) / "mounted"
    unmounted = Path(temporary) / "unmounted"
    mounted.mkdir(); unmounted.mkdir()
    test_events.to_parquet(mounted / "fault_events.parquet", index=False)
    pd.read_parquet(mounted / "fault_events.parquet")
    try:
        pd.read_parquet(unmounted / "fault_events.parquet")
    except FileNotFoundError:
        leakage_test = "pass"
    else:
        raise AssertionError("Leaky reader unexpectedly found unmounted truth")

single_detection = evaluate_alerts(
    perfect.iloc[[0]], test_events.iloc[[0]], test_intervals.iloc[[0]],
    exposure_value=10, exposure_unit="entity_day",
    decision_horizon_seconds=3600,
)
dense_alerts = pd.DataFrame([
    alert(f"D{minute:03d}", "asset-1", BASE + pd.Timedelta(minutes=minute))
    for minute in range(120)
], columns=ALERT_COLUMNS)
dense_detection = evaluate_alerts(
    dense_alerts, test_events.iloc[[0]], test_intervals.iloc[[0]],
    exposure_value=10, exposure_unit="entity_day",
    decision_horizon_seconds=3600,
)
assert single_detection["fault_results"]["detected"].sum() == 1
assert dense_detection["fault_results"]["detected"].sum() == 1
for recall_metric in (
    "event_recall", "preimpact_event_recall",
    "entity_fault_coverage",
):
    assert metric(single_detection, recall_metric) == metric(
        dense_detection, recall_metric
    )
point_adjustment_test = "pass"

tests = pd.DataFrame({
    "test": ["perfect", "constant_low", "constant_high", "recovery",
             "late", "duplicate", "invalid_alert", "leaky_reader",
             "no_point_adjustment"],
    "status": ["pass", "pass", "pass", "pass", "pass",
               "pass", invalid_alert_test, leakage_test,
               point_adjustment_test],
})
display(tests)

## 4. Freeze policy and physically separate holdout

In [ ]:
policy = {
    "evaluation_version": EVALUATION_VERSION,
    "evaluation_core_version": EVALUATION_CORE_VERSION,
    "sector": SECTOR,
    "canonical_fingerprint": manifest["fingerprint"],
    "primary_split": primary_split,
    **POLICY[SECTOR],
    "threshold_quantiles": [0.99, 0.995, 0.999],
    "persistence_observations": decisions["persistence_observations"],
    "recovery_observations": decisions["recovery_observations"],
    "matching_rule": "same affected entity; observable time to earlier of fault end and decision horizon",
    "duplicate_rule": "first alert receives event credit; later same-entity alerts are duplicates",
    "holdout_used": False,
}

with new_output_directory(OUTPUT_ROOT) as output:
    for name, folder in (("development", "development"), ("holdout", "holdout_sealed")):
        target = output / folder
        target.mkdir()
        tables = truth[name]
        tables["fault_events"].to_parquet(target / "fault_events.parquet", index=False)
        tables["fault_entity_intervals"].to_parquet(
            target / "fault_entity_intervals.parquet", index=False
        )
        if not tables["condition_states"].empty:
            tables["condition_states"].to_parquet(
                target / "condition_states.parquet", index=False
            )
    write_json(output / "evaluation_policy.json", policy)

display(pd.Series(policy, name="value").to_frame())
print("PASS — nine evaluation controls")
print("PASS — development and holdout truth are physically separate")
print("Saved:", OUTPUT_ROOT)
print("Next: 04_SIMPLE_ANOMALY_MODELS.ipynb")